# TN_test — cấu hình DS-TCN 192 kênh do nhóm tối ưu có đáng dùng không

## Đây là phép thăm dò, không phải một dòng của TN1

Chạy **một seed** để quyết định có đáng đầu tư ba seed hay không. Nếu đáng thì
mới nâng lên đủ ba seed và đưa vào bảng chính.

## Vì sao thăm dò

TN1 có một chỗ hổng thật, và hội đồng hỏi được:

> Mọi cấu hình TN1 chạy bằng siêu tham số của `optimal_params.json`, vốn được
> MobiVital dò riêng cho LSTM-352. Bảy kiến trúc tích chập đều chạy ở thiết lập
> tối ưu cho một kiến trúc khác. Vậy TCN thua thật, hay chỉ là chưa được chỉnh?

Nhóm đã tối ưu riêng một cấu hình DS-TCN cho bài toán này, và nó to hơn hẳn: **192 kênh, 4 khối**, khoảng 310 nghìn
tham số. Trên GHIJ nó đạt **0,8086 ± 0,0127**, ngang LSTM-352 (0,8103) với năm
lần ít tham số hơn, và hơn DS-TCN-64 của TN1 (0,7958).

Điểm GHIJ đó **sạch** — GHIJ chưa bao giờ dùng để chọn cấu hình ở cả hai bên.
Còn điểm trên KL thì **không dùng được**, vì cấu hình này được tối ưu
trên chính KL.

## Cấu hình cũ dựng lại được tới đâu

    nhóm tối ưu, RF121        kernel 5, 4 khối, 192 kênh, tầm nhìn 121
                              KHÔNG có lớp chuẩn hoá nào
                              310.873 tham số

    dựng lại bằng code này    ds_tcn --channels 192 --kernel_size 5
                              --n_blocks 4 --dropout 0.2
                              CÓ BatchNorm
                              313.945 tham số

Chênh đúng **3.072 = 8 lớp BatchNorm × 384**. Bỏ hết BatchNorm khỏi bản dựng
lại thì ra đúng 310.873. Đây là chỗ lệch duy nhất về kiến trúc, khoảng 1% tham
số, và có lý do giữ: `ds_tcn` của TN1 dùng BatchNorm theo Howard et al. 2017
mục 3.1.

Hai thứ của cấu hình đó **không** dựng lại được, vì `run_cv.py` chưa có cờ:

    learning rate    cũ 2,13e-4    ở đây 1e-4 theo giao thức TN1
    weight decay     cũ 3e-7       ở đây 0
    loss             cũ hỗn hợp    ở đây MSE thuần

Việc bỏ ba thứ đó là **có chủ ý**: giữ nguyên giao thức TN1 thì kết quả so thẳng
được với tám cấu hình kia. Chạy nguyên si cấu hình đó sẽ đổi sáu thứ cùng lúc,
thắng thua đều không quy được cho cái gì.

## Câu hỏi cụ thể

**DS-TCN thua có phải vì nó quá bé không?**

    DS-TCN-64      56.281 tham số     cv 0,7421 ± 0,0007
    DS-TCN-192    313.945 tham số     ?

Đổi ba thứ: số kênh 64 → 192, số khối 6 → 4, dropout 0 → 0,2. Không phải một
biến, nên nếu thắng thì chưa quy được cho riêng số kênh. Nhưng nếu **không**
thắng thì kết luận rõ ràng: sức chứa không phải thứ đang thiếu.

## Một chi tiết đáng ghi về tầm nhìn

    TN1 tcn/ds_tcn     kernel 3, 6 khối    tầm nhìn 253    phủ trọn 200 mẫu
    cũ RF121           kernel 5, 4 khối    tầm nhìn 121    thấy 121/200 mẫu
    cũ RF61            kernel 3, 4 khối    tầm nhìn  61    thấy  61/200 mẫu

Cả hai cấu hình đó đều **không nhìn hết cửa sổ đầu vào**. Vòng dò cũng
thấy điều đó: RF121 hơn RF61 trên GHIJ (0,8086 so với 0,7950). Có thể tầm nhìn
dài hơn nữa còn tốt hơn, nhưng quá trình tối ưu dừng ở 121.

## 1. Chuẩn bị Colab

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tải mã nguồn rồi vào thư mục đó.

In [2]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q --branch submission --single-branch https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : 0b08bac
commit MobiVital : 4319731 (đã ghim)
GPU              : Tesla T4, 15360 MiB


Lấy `by_user/` và `windows/` từ Drive.

In [3]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


## 2. Kiểm bản cài đặt và đối chiếu số tham số

Xác nhận hai điều trước khi train: model dựng đúng, và số tham số khớp con số
đã tính (313.945, tức 310.873 của bản nhóm tối ưu cộng 3.072 của BatchNorm).

In [4]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   52/52 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   313945

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   8 lớp BatchNorm1d, WeightNorm: False
   có BatchNorm như mong đợi                                  đạt
   không có WeightNorm                                        đạt

TẤT CẢ ĐẠT — bản cài đặt dùng được.


## 3. Chạy 4 fold, MỘT seed

Giao thức giữ nguyên TN1: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr` 0,9,
bốn fold cũ.

Tên cấu hình là `ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0`, nằm trong thư mục
`runs/tn_test/` riêng — không lẫn vào bảng TN1.

8 tầng tích chập ở 192 kênh, nặng hơn DS-TCN-64 khá nhiều. Ước lượng
**khoảng 1,5 giờ** cho bốn fold.

In [5]:
!python scripts/run_cv.py --experiment tn_test --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --seed 0

thực nghiệm tn_test  -> runs/tn_test/
cấu hình ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.04093  pearson 0.4171   1.3 phút
epoch  1  mse 0.02222  pearson 0.5340   2.7 phút
epoch  2  mse 0.02104  pearson 0.5536   4.1 phút
epoch  3  mse 0.02035  pearson 0.5653   5.5 phút
epoch  4  mse 0.01996  pearson 0.5728   6.9 phút
epoch  5  mse 0.01963  pearson 0.5771   8.3 phút
epoch  6  mse 0.01940  pearson 0.5816   9.7 phút
epoch  7  mse 0.01915  pearson 0.5842   11.1 phút
epoch  8  mse 0.01894  pearson 0.5880   12.5 phút
epoch  9  mse 0.01877  pearson 0.5901   13.9 phút
epoch 10  mse 0.01855  pearson 0.5938   15.3 phút
epoch 11  mse 0.01840  pearson 0.5963   16.7 phút
epoch 12  mse 0.01825  pearson 0.5978   18.2 phút
epoch 13  mse 0.01811  pearson 0.5984   19.6 phút
epoch 14  mse 0.01793  pearson 0.6015   21.0 phút
epoch 15  mse 0.01781  pearson 0.6032  

## 4. Cất kết quả

In [6]:
!python scripts/save_results.py tn_test --out tn_test_ds_tcn_c192

runs/tn_test/  ->  runs/tn_test_ds_tcn_c192.zip   (4.8 MB)
   5 dòng metric trong summary.csv

Bên trong:
        0  2026-09-07 08:00   tn_test/
        0  2026-09-07 05:56   tn_test/ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 06:35   tn_test/ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 07:15   tn_test/ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 07:53   tn_test/ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0_val_KL/
      161  2026-09-07 08:00   tn_test/README.txt
    16942  2026-09-07 08:00   tn_test/scores_ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0_val_KL.csv
    24342  2026-09-07 06:06   tn_test/scores_ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0_val_AB.csv
    19859  2026-09-07 07:23   tn_test/scores_ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0_val_DF.csv
     1420  2026-09-07 08:00   tn_test/summary.csv
    21714  2026-09-07 06:44   tn_test/scores_ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0_val_CE.csv
     1494  2

## 5. Đọc kết quả

`compare_cv` chỉ in được cấu hình trong `runs/tn_test/`, tức đúng một dòng. Để
so với TN1 thì lấy con số `cv_score` in ra ở đây rồi đặt cạnh bảng TN1.

Mốc để so:

| | tham số | cv_mean |
|---|---|---|
| LSTM-352 | 1.502.713 | 0,7570 |
| LSTM-67 | 56.908 | 0,7532 |
| DS-TCN-64 | 56.281 | 0,7421 |

Quyết định sau khi xem:

    hơn DS-TCN-64 rõ                      -> đáng chạy nốt 2 seed, đưa vào bảng
    ngang hoặc kém                        -> sức chứa không phải thứ đang thiếu,
                                             ghi kết quả âm này vào luận văn và
                                             chuyển sang TN3

In [7]:
!python scripts/compare_cv.py --experiment tn_test


BẢNG 1 — cv_score, thực nghiệm tn_test
cấu hình                         tham số  seed    cv_mean  seed_std  fold_std   từng seed
--------------------------------------------------------------------------------------------------------------
ds_tcn_c192_k5_n4_do0.2_mse_corr0.9    313945     1   0.756619       N/A  0.070195   s0 0.7566

cv_mean  = trung bình cv_score của các seed. cv_score của một seed là
           trung bình điểm macro 4 fold; macro = trung bình theo NGƯỜI.
seed_std = dao động giữa các seed. Chênh lệch giữa hai cấu hình nhỏ hơn
           số này thì chưa kết luận được.
fold_std = dao động giữa 4 fold, trung bình trên các seed. Nói dữ liệu
           giữa các người khác nhau ra sao, KHÔNG dùng để so cấu hình.



## 6. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()